In [1]:
import gzip
import json
from pathlib import Path
import pandas as pd

In [2]:
# Caminho dinâmico compatível com a estrutura de pastas do projeto
# Nota: No Jupyter Notebook, caso dê erro no __file__, você pode fixar o caminho base manualmente se preferir.
PASTA_PROJETO = Path.cwd().parent  # Sobe um nível a partir da pasta 'notebooks'

ARQUIVO_REVIEWS = (
    PASTA_PROJETO
    / "Data"
    / "steam_2025_5k-dataset-reviews_20250901.json.gz"
)

ARQUIVO_JOGOS = (
    PASTA_PROJETO
    / "Data"
    / "steam_2025_5k-dataset-games_20250831.json.gz"
)

print("Arquivo de reviews:", ARQUIVO_REVIEWS)
print("Arquivo existe?", ARQUIVO_REVIEWS.exists())

print("Arquivo de jogos:", ARQUIVO_JOGOS)
print("Arquivo existe?", ARQUIVO_JOGOS.exists())

Arquivo de reviews: c:\Users\Lenovo\OneDrive\Projeto Aplicado II\projeto-aplicado-II-classificacao-avaliacoes\Data\steam_2025_5k-dataset-reviews_20250901.json.gz
Arquivo existe? True
Arquivo de jogos: c:\Users\Lenovo\OneDrive\Projeto Aplicado II\projeto-aplicado-II-classificacao-avaliacoes\Data\steam_2025_5k-dataset-games_20250831.json.gz
Arquivo existe? True


In [3]:
with gzip.open(ARQUIVO_REVIEWS, "rt", encoding="utf-8") as arquivo:
    dados_reviews = json.load(arquivo)

print("\nArquivo de reviews carregado com sucesso!")
print("Chaves principais:", list(dados_reviews.keys()))


Arquivo de reviews carregado com sucesso!
Chaves principais: ['metadata', 'reviews']


In [4]:
linhas = []

for item in dados_reviews["reviews"]:
    appid = item["appid"]
    dados = item["review_data"]

    for review in dados["reviews"]:
        registro = review.copy()
        registro["appid"] = appid
        linhas.append(registro)

df_reviews = pd.DataFrame(linhas)

print(f"DataFrame criado com {len(df_reviews)} linhas e {len(df_reviews.columns)} colunas.")
display(df_reviews.head(3))

DataFrame criado com 37778 linhas e 18 colunas.


,recommendationid,author,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,primarily_steam_deck,appid,timestamp_dev_responded,developer_response
0,201194290,"{'steamid': '76561198152964416', 'num_games_ow...",russian,Я верю что однажды капитализм очиститься и люд...,1754033180,1754033180,True,58,2,0.834045231342315674,2,True,False,False,False,2210,NaN,NaN
1,202221261,"{'steamid': '76561198060071144', 'num_games_ow...",brazilian,Jogo é bom no geral. Mas uma pena não ter se q...,1755311121,1755311121,True,4,0,0.583333313465118408,0,True,False,False,False,2210,NaN,NaN
2,201240332,"{'steamid': '76561197965801053', 'num_games_ow...",english,This game feels like a mix of Quake 2+3 and so...,1754084409,1754084409,True,8,0,0.54731827974319458,0,True,False,False,False,2210,NaN,NaN


In [5]:
print("========================================")
print("DISTRIBUIÇÃO DAS RECOMENDAÇÕES")
print("========================================")

print(df_reviews["voted_up"].value_counts())

print("\nPercentual (%):")
display(
    df_reviews["voted_up"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

DISTRIBUIÇÃO DAS RECOMENDAÇÕES
voted_up
True     28546
False     9232
Name: count, dtype: int64

Percentual (%):


voted_up
True     75.56
False    24.44
Name: proportion, dtype: float64

In [6]:
print("========================================")
print("IDIOMAS MAIS FREQUENTES")
print("========================================0")

display(df_reviews["language"].value_counts().head(15))

IDIOMAS MAIS FREQUENTES
========================================0


language
english      19487
schinese      5209
russian       3756
german        1441
brazilian     1193
spanish       1073
japanese       932
french         927
koreana        873
turkish        650
tchinese       507
polish         467
latam          252
italian        240
czech          142
Name: count, dtype: int64

In [7]:
df_reviews["data"] = pd.to_datetime(
    df_reviews["timestamp_created"],
    unit="s"
)

print("========================================")
print("PERÍODO DAS AVALIAÇÕES")
print("========================================")
print("Primeira avaliação:", df_reviews["data"].min())
print("Última avaliação:", df_reviews["data"].max())

PERÍODO DAS AVALIAÇÕES
Primeira avaliação: 2010-11-20 20:38:46
Última avaliação: 2025-09-01 06:47:21


In [8]:
df_reviews["ano"] = df_reviews["data"].dt.year

print("========================================")
print("AVALIAÇÕES POR ANO")
print("========================================")

display(
    df_reviews["ano"]
    .value_counts()
    .sort_index()
)

AVALIAÇÕES POR ANO


ano
2010        5
2011       30
2012       43
2013      113
2014      656
2015      894
2016     1312
2017     2690
2018     2118
2019     2151
2020     2493
2021     2671
2022     2636
2023     2207
2024     2279
2025    15480
Name: count, dtype: int64

In [9]:
df_reviews["quantidade_palavras"] = (
    df_reviews["review"]
    .fillna("")
    .str.split()
    .str.len()
)

print("========================================")
print("ESTATÍSTICA DO TAMANHO DOS TEXTOS")
print("========================================")

display(df_reviews["quantidade_palavras"].describe())

ESTATÍSTICA DO TAMANHO DOS TEXTOS


count    37778.000000
mean        59.664964
std        116.518534
min          0.000000
25%          5.000000
50%         20.000000
75%         62.000000
max       3458.000000
Name: quantidade_palavras, dtype: float64

In [10]:
print("========================================")
print("MAPEAMENTO DE VALORES AUSENTES")
print("========================================")

display(
    df_reviews.isnull()
    .sum()
    .sort_values(ascending=False)
)

MAPEAMENTO DE VALORES AUSENTES


developer_response             36867
timestamp_dev_responded        36867
language                           0
author                             0
recommendationid                   0
timestamp_created                  0
review                             0
timestamp_updated                  0
voted_up                           0
weighted_vote_score                0
comment_count                      0
votes_up                           0
votes_funny                        0
received_for_free                  0
steam_purchase                     0
primarily_steam_deck               0
written_during_early_access        0
appid                              0
data                               0
ano                                0
quantidade_palavras                0
dtype: int64